# Stage 0: reproduce qualification attrition

All 833 wallets in the verified historical forecast-v4 score snapshot. This is an
engineering diagnostic, not a trading recommendation or a prospective backtest.
Set `MSOS_ATTRITION_SNAPSHOT` to the canonical enrichment JSONL from the verified
shadow run before executing. Run with Python 3.12 from this repository. The
notebook verifies source hashes and reproduces every committed analysis field.


In [1]:
import json
import os
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'services/polymarket-ingestor/src').is_dir())
sys.path.insert(0, str(root / 'services/polymarket-ingestor/src'))
from marketsignalos_polymarket.gate_attrition import build_report

snapshot = Path(os.environ['MSOS_ATTRITION_SNAPSHOT'])
benchmark = root / 'docs/benchmarks/2026-09-08-enrichment-shadow.json'
reference = json.loads((root / 'docs/benchmarks/2026-09-10-gate-attrition.json').read_text())
report = build_report(snapshot, benchmark)
assert report['analysis'] == reference['analysis']
assert report['provenance']['snapshot'] == reference['provenance']['snapshot']
assert report['provenance']['diagnostic_source_sha256'] == reference['provenance']['diagnostic_source_sha256']
print('Snapshot and all analysis fields match committed evidence.')
print('Snapshot SHA-256:', report['provenance']['snapshot']['sha256'])


Snapshot and all analysis fields match committed evidence.
Snapshot SHA-256: 01564d095e7945cee89895f3f278a0f0d66e9aee6b88530325f27e10a447dc5f


In [2]:
analysis = report['analysis']
print(json.dumps({key: analysis[key] for key in
                 ['wallets', 'trusted_wallets', 'tailable_wallets', 'counterfactuals',
                  'failure_groups', 'model_economic_failure_histogram']}, indent=2))


{
  "wallets": 833,
  "trusted_wallets": 125,
  "tailable_wallets": 0,
  "counterfactuals": {
    "data_gates_suspended": 2,
    "metadata_gate_suspended": 2,
    "clv_minimum_five": {
      "qualified_min": 0,
      "qualified_max": 0,
      "ambiguous_eligible_wallets": 0
    }
  },
  "failure_groups": {
    "coverage_only": 2,
    "model_or_economics_only": 125,
    "both": 706,
    "neither": 0
  },
  "model_economic_failure_histogram": {
    "0": 2,
    "1": 22,
    "2": 19,
    "3": 20,
    "4": 120,
    "5": 397,
    "6": 253
  }
}


In [3]:
print(json.dumps(analysis['waterfall'], indent=2))


[
  {
    "gate": 1,
    "label": "Complete activity history",
    "entering": 833,
    "removed": 11,
    "remaining": 822,
    "isolated_failures": 11,
    "not_evaluated": 0,
    "only_failure": 0
  },
  {
    "gate": 2,
    "label": "Complete current positions",
    "entering": 822,
    "removed": 3,
    "remaining": 819,
    "isolated_failures": 4,
    "not_evaluated": 0,
    "only_failure": 0
  },
  {
    "gate": 3,
    "label": "Complete closed positions",
    "entering": 819,
    "removed": 0,
    "remaining": 819,
    "isolated_failures": 1,
    "not_evaluated": 0,
    "only_failure": 0
  },
  {
    "gate": 4,
    "label": "Complete all-time economics",
    "entering": 819,
    "removed": 1,
    "remaining": 818,
    "isolated_failures": 2,
    "not_evaluated": 0,
    "only_failure": 0
  },
  {
    "gate": 5,
    "label": "Complete 30-day economics",
    "entering": 818,
    "removed": 12,
    "remaining": 806,
    "isolated_failures": 17,
    "not_evaluated": 0,
    "only_fai

## Interpretation

Two wallets fail only metadata completeness. Other failures overlap broadly;
removing data checks does not repair data or refit the model. Gate 12 is conditional
on sufficient recent sample. CLV-5 retains a positive lower bound, with ranges for
rounding ambiguity. Stored rejection reasons preserve pre-rounding decisions.
No thresholds or production data were changed. The next bounded investigation is
metadata-cause diagnosis; prospective, cost-adjusted following evidence is still
required. See the Markdown report and Stage 0 runbook for the full limitations.
